In [23]:
import pandas as pd
from transformers import pipeline
from transformers import RobertaModel, RobertaTokenizer, RobertaForSequenceClassification
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

In [34]:
# Load the ACLED notes
notes = pd.read_csv("test_notes.csv")

In [25]:
# Define a custom to function to run the classifier

def get_labels(notes_with_ids, model, candidate_labels):
    results = []

    classifier = pipeline("zero-shot-classification", model=model)

    for index, row in notes_with_ids.iterrows():
        event_id = row['event_id']
        note = row['note']  
        sequence_to_classify = note
        output = classifier(sequence_to_classify, candidate_labels, multi_label=False)
        scores_for_note = {label: score for label, score in zip(output["labels"], output["scores"])}
        scores_for_note['Event ID'] = event_id
        results.append(scores_for_note)

    df = pd.DataFrame(results)
    df.set_index('Event ID', inplace=True)
    return df




In [26]:
# Define the model
model = "MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli"

# Define the classifier pipeline
classifier = pipeline("zero-shot-classification", model = "MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli")

# Create the topics to classify on
candidate_labels = ["environment", "unions", "gun-violence", "healthcare", "racial-justice"]

In [35]:
# Run the classifier function on the ACLED notes, the specified model, and the defined topic labels
topics = get_labels(notes, model, candidate_labels)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


In [36]:
topics = topics.apply(lambda row: row == row.max(), axis=1).astype(int)

In [37]:
# Add categotical label variable to the topics dataframe
topics['label'] = topics.idxmax(axis=1)

# Add additional numerical label variable to the topics dataframe
topics['label_num'] = topics['label'].map({'environment': 1, 'unions': 2, 'gun-violence': 3, 'healthcare': 4, 'racial-justice': 5})


In [22]:
# Save the results to a CSV file
topics.to_csv("topics.csv")